# Module 7.1: Industrial Experiment Tracking & Governance with MLflow
### 🇵🇱 Śledzenie Eksperymentów i Zarządzanie Modelami Przemysłowymi w MLflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m07_l01_experiment_tracking_mlflow.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Establish rigorous MLOps experiment tracking and audit trails: log metrics, artifacts, and manage the Model Registry lifecycle.  
> 🇵🇱 **Streszczenie:** Wdróż przemysłowy system MLOps: śledź metryki, rejestruj artefakty i zarządzaj cyklem życia modeli w rejestrze MLflow.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Auditable Manufacturing MLOps**: In regulated plants (ISO 9001), automated decisions must be backed by documented training parameters, dataset hashes, and metric logs.
2. **MLflow Tracking**: Logs hyperparameters, evaluation metrics (ROC-AUC, financial losses), and serialized model binaries.
3. **Model Registry Stages**: Lifecycle gates: `Staging` (shadow testing on live PLC tags) $	o$ `Production` $	o$ `Archived` (fallback).


In [ ]:
# 1. Simulate MLflow Experiment Tracking Run Matrix
np.random.seed(42)
n_experiments = 8

# Model hyperparameter experiments across 8 training iterations
experiments = []
for i in range(n_experiments):
    n_est = int(np.random.choice([50, 100, 150, 200]))
    lr = float(np.random.choice([0.01, 0.05, 0.1, 0.2]))
    max_d = int(np.random.choice([3, 4, 5, 6]))
    
    # Simulated validation performance metrics
    auc = round(0.88 + 0.08 * (1.0 - lr) + np.random.normal(0, 0.01), 4)
    auc = min(0.99, max(0.85, auc))
    # Financial expected loss: higher AUC = lower cost
    cost_loss = round((1.0 - auc) * 45000 + np.random.normal(0, 500), 2)
    
    experiments.append({
        "run_id": f"run_{i+1:02d}",
        "n_estimators": n_est,
        "learning_rate": lr,
        "max_depth": max_d,
        "roc_auc": auc,
        "expected_loss_usd": cost_loss
    })

df_mlflow = pd.DataFrame(experiments).sort_values("expected_loss_usd").reset_index(drop=True)
print("MLflow Logged Production Runs:")
print(df_mlflow)


In [ ]:
# 2. Visualize MLflow Run Metrics & Model Registry Staging
best_run = df_mlflow.iloc[0]

plt.figure(figsize=(10, 4.5))
plt.scatter(df_mlflow["roc_auc"], df_mlflow["expected_loss_usd"], color='#0284c7', s=100, label='Candidate Model Runs')
plt.scatter(best_run["roc_auc"], best_run["expected_loss_usd"], color='#10b981', s=200, marker='*', label=f'Promoted to Production ({best_run["run_id"]})')

plt.title("MLflow Model Evaluation: ROC-AUC vs Financial Expected Loss", fontweight='bold')
plt.xlabel("Validation ROC-AUC")
plt.ylabel("Expected Financial Downtime Loss [USD]")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Promoting Model {best_run['run_id']} to 'Production' stage (Loss: ${best_run['expected_loss_usd']:,.2f})")


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Initialize MLflow tracking server and create dedicated manufacturing experiment *(🇵🇱 Zainicjalizuj serwer śledzenia MLflow i utwórz dedykowany eksperyment produkcyjny)*
- [ ] **Task 2:** Log hyperparameters, financial loss matrices, and serialized model artifacts *(🇵🇱 Zarejestruj hiperparametry, macierze strat finansowych i zserializowane artefakty)*
- [ ] **Task 3:** Promote candidate model across Model Registry stages: Staging to Production *(🇵🇱 Przeprowadź model przez etapy rejestru modeli: od Staging do Production)*


In [ ]:
# Exercise Challenge:
# Assert that the promoted model has ROC-AUC > 0.95 and lowest expected financial loss
assert best_run["roc_auc"] > 0.95, "Promoted model failed ROC-AUC threshold."
assert best_run["expected_loss_usd"] == df_mlflow["expected_loss_usd"].min(), "Best model was not selected."
print("✓ MLflow governance and model registry verified.")
